# two_stage_reentry_derisk_purged_bc_v1

Purpose: continue from `reentry_label_conditioned_bc_diagnostic_v1`, where a single logistic classifier could not learn the future-aware dual oracle safely. This notebook tests a more explicit two-stage design: first classify forward drawdown risk, then allow re-entry labels only when the de-risk classifier says risk has cleared.

Governance:
- Baseline: `current_runtime_paper` daily replay from `runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv`.
- Direct comparison: same replay with predicted two-stage labels adjusting daily `position`.
- Role: `diagnose`, no actor checkpoint and no promotion.
- Changed layer: `model_selection` / validation design. Reusable split and safe scoring helpers live in `src.soxl_quant.app.purged_walkforward`; grid logic stays in this notebook.
- Target windows: `wf_2022`, `wf_2023_2024`, C002 2025 rebound, and C003 2026 re-entry.
- Accepted trade-off: a candidate must reach mean window delta return `>= +30pp` and every scored key-window MaxDD `<= 15%`; otherwise it remains diagnostic.

Historical reflection:
- Closest success: future-aware dual oracle in `reentry_label_conditioned_bc_diagnostic_v1` shows the label family can reach sub-15% drawdown, but only with future leakage.
- Closest failure: the same notebook's one-shot no-leak logistic BC failed both return and drawdown gates.
- Why this is not a duplicate: this run uses purged expanding splits and two separate stage classifiers, so de-risk timing is learned before re-entry timing rather than forced through one multiclass classifier.

External references used as design constraints:
- Qlib paper / GitHub for keeping the workflow inside a reproducible quant harness: https://arxiv.org/abs/2009.11189 and https://github.com/microsoft/qlib
- Deflated Sharpe and PBO for selection-bias discipline: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551 and https://core.ac.uk/display/24041876
- Purged/embargoed financial CV from Advances in Financial Machine Learning, reflected in public summaries: https://www.oreilly.com/library/view/advances-in-financial/9781119482086/c07.xhtml
- Behavior cloning in trading-agent settings as diagnostic target design, not promotion proof: https://arxiv.org/abs/2110.01325


In [1]:
from __future__ import annotations

import json
from dataclasses import asdict
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.soxl_quant.app.budget_counterfactual import compute_metrics
from src.soxl_quant.app.oos_diagnostics import load_qlib_feature_frame
from src.soxl_quant.app.purged_walkforward import build_purged_walkforward_splits, single_position_frame
from src.soxl_quant.app.reentry_labels import ReentryOracleLabelConfig, build_reentry_oracle_label_frame

EXPERIMENT = "two_stage_reentry_derisk_purged_bc_v1"
OUTPUT_DIR = Path("runtime/model/card") / EXPERIMENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DAILY_CSV = Path("runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv")
DAILY = pd.read_csv(DAILY_CSV, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)
PRICE = DAILY.set_index("datetime")["price"].astype(float)
FEATURES = load_qlib_feature_frame(str(DAILY["datetime"].min().date()), str(DAILY["datetime"].max().date()))
FEATURES = FEATURES.reset_index().rename(columns={"index": "datetime"})
FEATURES["datetime"] = pd.to_datetime(FEATURES["datetime"])

WINDOWS = {
    "wf_2022": ("2022-01-01", "2022-12-31"),
    "wf_2023_2024": ("2023-01-01", "2024-12-31"),
    "bad_C002_2025_rebound": ("2025-04-04", "2025-07-22"),
    "bad_C003_2026_reentry": ("2026-02-04", "2026-05-21"),
}

FEATURE_COLS = [
    "vol_of_vol_21", "tracking_error_vol_21", "decay_63", "realized_vol_63", "tracking_error_21",
    "realized_vol_21", "mom_21", "mom_10", "relative_strength_21", "vix_percentile_252",
    "anchor_gap_180", "below_anchor_ratio_63", "regime_risk", "drawdown_63", "downside_semivol_21",
    "vix_zscore", "trend_gap_21", "trend_gap_63", "realized_vol_10", "days_since_high_63",
]

print({
    "daily_rows": len(DAILY),
    "daily_start": str(DAILY["datetime"].min().date()),
    "daily_end": str(DAILY["datetime"].max().date()),
    "feature_rows": len(FEATURES),
    "feature_cols": len(FEATURE_COLS),
})

[69553:MainThread](2026-05-25 12:49:33,506) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[69553:MainThread](2026-05-25 12:49:34,164) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[69553:MainThread](2026-05-25 12:49:34,165) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[69553:MainThread](2026-05-25 12:49:36,656) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[69553:MainThread](2026-05-25 12:49:36,658) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[69553:MainThread](2026-05-25 12:49:36,659) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | Init data Done


{'daily_rows': 1605, 'daily_start': '2020-01-02', 'daily_end': '2026-05-21', 'feature_rows': 1605, 'feature_cols': 20}


In [2]:
def score_positions(frame: pd.DataFrame, start: str, end: str, position_col: str = "position") -> dict:
    sub = frame[(frame["datetime"] >= start) & (frame["datetime"] <= end)].copy()
    return compute_metrics(single_position_frame(sub, position_col=position_col), PRICE.loc[start:end])

def base_window_table() -> pd.DataFrame:
    rows = []
    for name, (start, end) in WINDOWS.items():
        rows.append({"window": name, "variant": "baseline", **score_positions(DAILY, start, end)})
    return pd.DataFrame(rows)

BASE_WINDOWS = base_window_table()
BASE_WINDOWS[["window", "return", "relative_return", "max_dd", "avg_pos", "trades"]]

,window,return,relative_return,max_dd,avg_pos,trades
0,wf_2022,-21.567182,65.020890,31.096849,0.248277,26
1,wf_2023_2024,209.141657,17.368148,30.668184,0.497231,70
2,bad_C002_2025_rebound,76.882481,-124.606649,16.324924,0.572851,9
3,bad_C003_2026_reentry,17.799626,-217.582972,6.421473,0.161722,2


In [3]:
def build_labeled_feature_frame(
    cfg: ReentryOracleLabelConfig,
    *,
    derisk_dd: float,
    derisk_loss: float = -0.05,
) -> pd.DataFrame:
    labels = build_reentry_oracle_label_frame(DAILY, cfg)
    labels["derisk_label"] = (
        labels["has_forward_window"]
        & (labels["base_position"] >= 0.05)
        & ((labels["forward_drawdown"] >= float(derisk_dd)) | (labels["forward_return"] <= float(derisk_loss)))
    )
    # Re-entry is deliberately staged after de-risk; a row cannot teach both actions.
    labels["staged_reentry_label"] = labels["oracle_reentry"] & (~labels["derisk_label"])
    frame = labels.merge(FEATURES[["datetime", *FEATURE_COLS]], on="datetime", how="left")
    return frame.dropna(subset=FEATURE_COLS).reset_index(drop=True)

def binary_probability(train: pd.DataFrame, test: pd.DataFrame, label_col: str, *, c_value: float) -> np.ndarray:
    y = train[label_col].astype(int)
    if y.nunique() == 0:
        return np.zeros(len(test), dtype=float)
    if y.nunique() == 1:
        return np.full(len(test), float(y.iloc[0]), dtype=float)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=float(c_value), class_weight="balanced", max_iter=1000, random_state=7),
    )
    model.fit(train[FEATURE_COLS], y)
    classes = list(model.named_steps["logisticregression"].classes_)
    proba = model.predict_proba(test[FEATURE_COLS])
    if 1 not in classes:
        return np.zeros(len(test), dtype=float)
    return proba[:, classes.index(1)]

def apply_two_stage_path(
    test: pd.DataFrame,
    *,
    cfg: ReentryOracleLabelConfig,
    p_derisk: np.ndarray,
    p_reentry: np.ndarray,
    derisk_threshold: float,
    clear_threshold: float,
    reentry_threshold: float,
    derisk_target: float,
) -> pd.DataFrame:
    out = test[["datetime", "base_position"]].rename(columns={"base_position": "position"}).copy()
    derisk_mask = (p_derisk >= float(derisk_threshold)) & (test["base_position"].to_numpy(dtype=float) >= 0.05)
    reentry_mask = (
        (p_derisk <= float(clear_threshold))
        & (p_reentry >= float(reentry_threshold))
        & (test["base_position"].to_numpy(dtype=float) <= float(cfg.max_base_position))
        & (~derisk_mask)
    )
    out.loc[derisk_mask, "position"] = np.minimum(out.loc[derisk_mask, "position"], float(derisk_target))
    out.loc[reentry_mask, "position"] = np.maximum(out.loc[reentry_mask, "position"], float(cfg.target_position))
    out["p_derisk"] = p_derisk
    out["p_reentry"] = p_reentry
    out["derisk_pred"] = derisk_mask
    out["reentry_pred"] = reentry_mask
    return out

def score_candidate(
    *,
    cfg: ReentryOracleLabelConfig,
    derisk_dd: float,
    derisk_loss: float,
    derisk_threshold: float,
    clear_threshold: float,
    reentry_threshold: float,
    derisk_target: float,
    c_derisk: float,
    c_reentry: float,
) -> tuple[dict, list[dict], dict[str, pd.DataFrame]]:
    frame = build_labeled_feature_frame(cfg, derisk_dd=derisk_dd, derisk_loss=derisk_loss)
    splits = build_purged_walkforward_splits(
        frame,
        WINDOWS,
        purge_rows=int(cfg.horizon_days),
        min_train_rows=100,
        min_test_rows=2,
    )
    row = {
        **{f"cfg_{k}": v for k, v in asdict(cfg).items()},
        "derisk_dd": float(derisk_dd),
        "derisk_loss": float(derisk_loss),
        "derisk_threshold": float(derisk_threshold),
        "clear_threshold": float(clear_threshold),
        "reentry_threshold": float(reentry_threshold),
        "derisk_target": float(derisk_target),
        "c_derisk": float(c_derisk),
        "c_reentry": float(c_reentry),
        "split_count": len(splits),
        "total_derisk_predictions": 0,
        "total_reentry_predictions": 0,
    }
    comparison_rows = []
    path_by_window: dict[str, pd.DataFrame] = {}
    deltas = []
    max_dds = []
    for split in splits:
        train = split.train[split.train["has_forward_window"]].copy()
        test = split.test.copy()
        p_derisk = binary_probability(train, test, "derisk_label", c_value=c_derisk)
        reentry_train = train[~train["derisk_label"]].copy()
        p_reentry = binary_probability(reentry_train, test, "staged_reentry_label", c_value=c_reentry)
        path = apply_two_stage_path(
            test,
            cfg=cfg,
            p_derisk=p_derisk,
            p_reentry=p_reentry,
            derisk_threshold=derisk_threshold,
            clear_threshold=clear_threshold,
            reentry_threshold=reentry_threshold,
            derisk_target=derisk_target,
        )
        start, end = WINDOWS[split.name]
        metrics = score_positions(path, start, end)
        base_metrics = score_positions(test[["datetime", "base_position"]].rename(columns={"base_position": "position"}), start, end)
        delta_return = metrics["return"] - base_metrics["return"]
        row[f"{split.name}_return"] = metrics["return"]
        row[f"{split.name}_relative_return"] = metrics["relative_return"]
        row[f"{split.name}_delta_return"] = delta_return
        row[f"{split.name}_max_dd"] = metrics["max_dd"]
        row[f"{split.name}_delta_max_dd"] = metrics["max_dd"] - base_metrics["max_dd"]
        row[f"{split.name}_avg_pos"] = metrics["avg_pos"]
        row[f"{split.name}_derisk_predictions"] = int(path["derisk_pred"].sum())
        row[f"{split.name}_reentry_predictions"] = int(path["reentry_pred"].sum())
        row["total_derisk_predictions"] += int(path["derisk_pred"].sum())
        row["total_reentry_predictions"] += int(path["reentry_pred"].sum())
        deltas.append(delta_return)
        max_dds.append(metrics["max_dd"])
        comparison_rows.extend([
            {"window": split.name, "variant": "baseline", **base_metrics},
            {"window": split.name, "variant": "two_stage", **metrics},
        ])
        path_by_window[split.name] = path
    row["mean_delta_return"] = float(np.mean(deltas)) if deltas else float("nan")
    row["min_delta_return"] = float(np.min(deltas)) if deltas else float("nan")
    row["max_key_window_dd"] = float(np.max(max_dds)) if max_dds else float("nan")
    row["return_gate"] = bool(row["mean_delta_return"] >= 30.0 and row["min_delta_return"] >= -10.0) if deltas else False
    row["maxdd_gate"] = bool(row["max_key_window_dd"] <= 15.0) if max_dds else False
    row["full_gate"] = bool(row["return_gate"] and row["maxdd_gate"] and len(splits) == len(WINDOWS))
    row["diagnostic_score"] = (
        row["mean_delta_return"]
        - 3.5 * max(row["max_key_window_dd"] - 15.0, 0.0)
        + 0.25 * row["min_delta_return"]
    )
    return row, comparison_rows, path_by_window

## Grid

This is intentionally smaller than a training sweep. It varies label horizon, target position, drawdown label severity, regularization, and action thresholds. Each candidate uses expanding train data before the window and purges `horizon_days` rows immediately before the test window.

In [4]:
GRID_CONFIGS = [
    ReentryOracleLabelConfig(21, 0.65, 0.75, 0.40, 0.18),
    ReentryOracleLabelConfig(21, 0.55, 0.75, 0.40, 0.18),
    ReentryOracleLabelConfig(10, 0.65, 0.75, 0.40, 0.10),
    ReentryOracleLabelConfig(21, 0.65, 0.65, 0.25, 0.15),
]
rows = []
comparison_by_key = {}
path_by_key = {}
for cfg, derisk_dd, c_derisk, c_reentry, derisk_threshold, reentry_threshold in product(
    GRID_CONFIGS,
    [0.10, 0.15],
    [0.05, 0.20],
    [0.05, 0.20],
    [0.55, 0.65, 0.75],
    [0.55, 0.65, 0.75],
):
    clear_threshold = min(float(derisk_threshold), 0.55)
    row, comparison_rows, paths = score_candidate(
        cfg=cfg,
        derisk_dd=derisk_dd,
        derisk_loss=-0.05,
        derisk_threshold=derisk_threshold,
        clear_threshold=clear_threshold,
        reentry_threshold=reentry_threshold,
        derisk_target=0.0,
        c_derisk=c_derisk,
        c_reentry=c_reentry,
    )
    key = len(rows)
    row["candidate_key"] = key
    rows.append(row)
    comparison_by_key[key] = comparison_rows
    path_by_key[key] = paths

results = pd.DataFrame(rows).sort_values(["full_gate", "diagnostic_score", "mean_delta_return"], ascending=[False, False, False]).reset_index(drop=True)
results.to_csv(OUTPUT_DIR / "two_stage_grid_results.csv", index=False)
results.head(12)[[
    "candidate_key", "cfg_horizon_days", "cfg_max_base_position", "cfg_target_position", "cfg_min_forward_return",
    "derisk_dd", "derisk_threshold", "clear_threshold", "reentry_threshold", "c_derisk", "c_reentry",
    "mean_delta_return", "min_delta_return", "max_key_window_dd", "return_gate", "maxdd_gate", "full_gate",
    "total_derisk_predictions", "total_reentry_predictions", "diagnostic_score",
]]

,candidate_key,cfg_horizon_days,cfg_max_base_position,cfg_target_position,cfg_min_forward_return,derisk_dd,derisk_threshold,clear_threshold,reentry_threshold,c_derisk,c_reentry,mean_delta_return,min_delta_return,max_key_window_dd,return_gate,maxdd_gate,full_gate,total_derisk_predictions,total_reentry_predictions,diagnostic_score
0,125,21,0.55,0.75,0.40,0.15,0.75,0.55,0.75,0.05,0.20,18.197413,0.000000,30.668184,False,False,False,7,54,-36.641231
1,259,21,0.65,0.65,0.25,0.15,0.75,0.55,0.65,0.05,0.05,16.410728,0.000000,30.668184,False,False,False,7,34,-38.427916
2,2,21,0.65,0.75,0.40,0.10,0.55,0.55,0.75,0.05,0.05,-1.656277,-43.775957,22.525921,False,False,False,141,19,-38.940989
3,116,21,0.55,0.75,0.40,0.15,0.75,0.55,0.75,0.05,0.05,15.278657,0.000000,30.668184,False,False,False,7,32,-39.559987
4,53,21,0.65,0.75,0.40,0.15,0.75,0.55,0.75,0.05,0.20,14.322161,0.000000,30.668184,False,False,False,7,65,-40.516484
5,269,21,0.65,0.65,0.25,0.15,0.75,0.55,0.75,0.05,0.20,12.283103,0.000000,30.668184,False,False,False,7,29,-42.555541
6,44,21,0.65,0.75,0.40,0.15,0.75,0.55,0.75,0.05,0.05,10.289673,-0.452495,30.668184,False,False,False,7,34,-44.662095
7,124,21,0.55,0.75,0.40,0.15,0.75,0.55,0.65,0.05,0.20,12.538437,0.000000,31.387220,False,False,False,7,85,-44.816834
8,143,21,0.55,0.75,0.40,0.15,0.75,0.55,0.75,0.20,0.20,5.784003,-4.580489,30.668184,False,False,False,38,44,-50.199764
9,23,21,0.65,0.75,0.40,0.10,0.65,0.55,0.75,0.20,0.05,-4.444530,-47.045680,25.425133,False,False,False,107,18,-52.693914


In [5]:
selected = results.iloc[0].to_dict()
selected_key = int(selected["candidate_key"])
selected_comparison = pd.DataFrame(comparison_by_key[selected_key])
selected_comparison.to_csv(OUTPUT_DIR / "selected_comparison.csv", index=False)

# Save selected per-window path rows for audit, but keep notebook as the primary conclusion.
selected_paths = []
for window, frame in path_by_key[selected_key].items():
    out = frame.copy()
    out["window"] = window
    selected_paths.append(out)
if selected_paths:
    pd.concat(selected_paths, ignore_index=True).to_csv(OUTPUT_DIR / "selected_paths.csv", index=False)

selection = {
    "experiment": EXPERIMENT,
    "decision": "promote_candidate" if bool(selected["full_gate"]) else "diagnostic_not_promoted",
    "reason": (
        "two-stage purged classifier clears return and drawdown gates"
        if bool(selected["full_gate"])
        else "two-stage purged classifier still fails the combined +30pp mean return and <=15% key-window MaxDD gates"
    ),
    "baseline_daily_csv": str(DAILY_CSV),
    "grid_results_csv": str(OUTPUT_DIR / "two_stage_grid_results.csv"),
    "selected_comparison_csv": str(OUTPUT_DIR / "selected_comparison.csv"),
    "selected_paths_csv": str(OUTPUT_DIR / "selected_paths.csv"),
    "selected_candidate": {k: (None if pd.isna(v) else v) for k, v in selected.items() if isinstance(v, (int, float, str, bool)) or pd.isna(v)},
    "next_path": (
        "package for stricter saved-actor walk-forward validation"
        if bool(selected["full_gate"])
        else "do not open PPO retrain from this label yet; inspect false positive de-risk windows and consider a tree/sequence model or stronger drawdown-state features"
    ),
}
(OUTPUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2, ensure_ascii=True), encoding="utf-8")
selected_comparison[["window", "variant", "return", "relative_return", "max_dd", "avg_pos", "trades"]].sort_values(["window", "variant"])

,window,variant,return,relative_return,max_dd,avg_pos,trades
4,bad_C002_2025_rebound,baseline,76.882481,-124.606649,16.324924,0.572851,9
5,bad_C002_2025_rebound,two_stage,105.987796,-95.501335,12.478720,0.531770,10
6,bad_C003_2026_reentry,baseline,17.799626,-217.582972,6.421473,0.161722,2
7,bad_C003_2026_reentry,two_stage,17.799626,-217.582972,6.421473,0.161722,2
0,wf_2022,baseline,-21.567182,65.020890,31.096849,0.248277,26
1,wf_2022,two_stage,3.945751,90.533823,26.744475,0.326961,40
2,wf_2023_2024,baseline,209.141657,17.368148,30.668184,0.497231,70
3,wf_2023_2024,two_stage,227.313063,35.539553,30.668184,0.514590,76


In [6]:
best = results.iloc[0]
pass_count = int(results["full_gate"].sum())
summary = {
    "candidate_count": int(len(results)),
    "full_gate_pass_count": pass_count,
    "best_mean_delta_return": float(best["mean_delta_return"]),
    "best_min_delta_return": float(best["min_delta_return"]),
    "best_max_key_window_dd": float(best["max_key_window_dd"]),
    "best_total_derisk_predictions": int(best["total_derisk_predictions"]),
    "best_total_reentry_predictions": int(best["total_reentry_predictions"]),
}
conclusion = f"""## Runtime Conclusion

Decision: `{selection['decision']}`.

Screened `{summary['candidate_count']}` two-stage purged BC candidates; full gate pass count `{summary['full_gate_pass_count']}`. Best candidate mean window delta `{summary['best_mean_delta_return']:.2f}pp`, min window delta `{summary['best_min_delta_return']:.2f}pp`, and max key-window DD `{summary['best_max_key_window_dd']:.2f}%`.

Mechanism readout: staging de-risk before re-entry reduces some one-shot multiclass noise, but the selected candidate still cannot get every key window under the `15%` drawdown target. The remaining failure is not broad under-participation alone; false positive/late de-risk timing still dominates the risk gate.

Keep/drop: keep the reusable purged split and `single_position_frame` helpers. Do not promote this two-stage logistic label policy and do not start a broad PPO retrain from it. The next valid experiment should either use a non-linear tree/sequence learner for the same purged labels or add stronger drawdown-state features before testing PPO warm-start.

Artifacts: `two_stage_grid_results.csv`, `selected_comparison.csv`, `selected_paths.csv`, and `selection.json`.
"""
display(Markdown(conclusion))
print(json.dumps(summary, indent=2))

## Runtime Conclusion

Decision: `diagnostic_not_promoted`.

Screened `288` two-stage purged BC candidates; full gate pass count `0`. Best candidate mean window delta `18.20pp`, min window delta `0.00pp`, and max key-window DD `30.67%`.

Mechanism readout: staging de-risk before re-entry reduces some one-shot multiclass noise, but the selected candidate still cannot get every key window under the `15%` drawdown target. The remaining failure is not broad under-participation alone; false positive/late de-risk timing still dominates the risk gate.

Keep/drop: keep the reusable purged split and `single_position_frame` helpers. Do not promote this two-stage logistic label policy and do not start a broad PPO retrain from it. The next valid experiment should either use a non-linear tree/sequence learner for the same purged labels or add stronger drawdown-state features before testing PPO warm-start.

Artifacts: `two_stage_grid_results.csv`, `selected_comparison.csv`, `selected_paths.csv`, and `selection.json`.


{
  "candidate_count": 288,
  "full_gate_pass_count": 0,
  "best_mean_delta_return": 18.197413391958435,
  "best_min_delta_return": 0.0,
  "best_max_key_window_dd": 30.66818414624621,
  "best_total_derisk_predictions": 7,
  "best_total_reentry_predictions": 54
}
